# fake_pico.py
**網路、多人遊戲與 Web 全端 › 🚀 延伸:MQTT 捷徑路線(聊天室、多人遊戲、Pico 即時儀表板)**



> 🎧 有逐段旁白、逐句拆解的版本在 [pyl Academy](https://friencdking0000.github.io/pyl-academy/#/learn/PY-NETWEB/7/5)。
>
> **怎麼用**:由上往下,每格按左邊 ▶ 執行;或選單「執行階段 › 全部執行」。

In [ ]:
#@title 0. 準備環境(下載課程範例檔、安裝套件)
!pip -q install paho-mqtt
import sys, os; sys.argv = sys.argv[:1]  # 讓 argparse / sys.argv 在 notebook 裡也正常
__file__ = os.path.join(os.getcwd(), 'fake_pico.py')  # notebook 沒有 __file__,程式靠它找同資料夾的檔案

### 沒有 Pico 也能玩:用 Python 假裝成一片 Pico,發布「模擬的」M
沒有 Pico 也能玩:用 Python 假裝成一片 Pico,發布「模擬的」MPU6050 數據。格式跟 pico2w_mqtt_mpu6050.ino 一模一樣 → dashboard.html 分不出真假。安裝:pip install paho-mqtt。執行:python fake_pico.py [房間名稱]。

In [ ]:
# 沒有 Pico 也能玩:用 Python 假裝成一片 Pico,發布「模擬的」MPU6050 數據

### import json
這一段從「import json」開始,先看懂每一行在做什麼,再按執行看看結果。

In [ ]:
# 格式跟 pico2w_mqtt_mpu6050.ino 一模一樣 → dashboard.html 分不出真假
# 安裝:pip install paho-mqtt
# 執行:python fake_pico.py [房間名稱]

import json
import math
import random
import sys
import time

### import paho.mqtt.client as mqtt
這一段從「import paho.mqtt.client as mqtt」開始,先看懂每一行在做什麼,再按執行看看結果。

In [ ]:
import paho.mqtt.client as mqtt

ROOM   = sys.argv[1] if len(sys.argv) > 1 else "demo"
DEVICE = "fake-" + "".join(random.choice("0123456789abcdef") for _ in range(4))
TOPIC  = f"pyl-academy/v1/{ROOM}/pico/{DEVICE}/"

### client = mqtt.Client(mqtt.CallbackAPIVer
這一段從「client = mqtt.Client(mqtt.CallbackAPIVersion.VERSI」開始,先看懂每一行在做什麼,再按執行看看結果。

In [ ]:
client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2, client_id=DEVICE)
client.will_set(TOPIC + "status", "offline", retain=True)       # 關掉程式 → 網頁顯示離線
client.connect("broker.emqx.io", 1883, keepalive=15)
client.loop_start()
client.publish(TOPIC + "status", "online", retain=True)
print(f"模擬裝置 {DEVICE} 開始發布到房間「{ROOM}」,打開 dashboard.html?room={ROOM} 看曲線(Ctrl+C 結束)")

### 假裝板子在桌上慢慢搖晃:z 軸約 1g(重力),x、y 小幅擺動,再加一點雜訊
假裝板子在桌上慢慢搖晃:z 軸約 1g(重力),x、y 小幅擺動,再加一點雜訊。

In [ ]:
t0 = time.time()
try:
    while True:
        t = time.time() - t0
        # 假裝板子在桌上慢慢搖晃:z 軸約 1g(重力),x、y 小幅擺動,再加一點雜訊
        data = {
            "ax": round(0.3 * math.sin(t * 1.5) + random.gauss(0, 0.02), 3),
            "ay": round(0.2 * math.sin(t * 0.7 + 1) + random.gauss(0, 0.02), 3),
            "az": round(1 + random.gauss(0, 0.02), 3),
            "gx": round(45 * math.cos(t * 1.5), 1),
            "gy": round(14 * math.cos(t * 0.7 + 1), 1),
            "gz": round(random.gauss(0, 1), 1),
            "t": int(t * 1000),
        }
        client.publish(TOPIC + "imu", json.dumps(data))
        time.sleep(0.1)                                         # 每秒 10 筆,跟真的 Pico 一樣
except KeyboardInterrupt:
    pass
finally:
    client.publish(TOPIC + "status", "offline", retain=True)
    time.sleep(0.3)
    client.loop_stop()
    client.disconnect()